
# 練習問題: 配列 (動的割り当て)

## 目標

要素数を **実行時に** 決める動的配列の確保のしかたを身につける.
C++ は `malloc`/`free` とポインタ, Fortran は `allocatable` と `allocate`/`deallocate` を使う.

## 課題

`array_dynamic.cpp` (または `array_dynamic.f90`) は, 要素数 `n` をコマンドライン引数で受け取り, `1/1 + 1/2 + … + 1/n` を計算する.
配列の確保 (メモリの割り当て) が抜けているので, このままでは正しく動かない (実行時にエラーになる).

`TODO` の箇所に **配列を確保する処理** を書け.

- C++: `a = (double *)malloc(sizeof(double) * n);`
- Fortran: `allocate(a(n))`

(解放 `free(a)` / `deallocate(a)` は最後にすでに書いてある.)

## コンパイルと実行

```
# C++
nvc++ -fast array_dynamic.cpp -o array_dynamic.exe

# Fortran
nvfortran -fast array_dynamic.f90 -o array_dynamic.exe
```

```
./array_dynamic.exe 100
./array_dynamic.exe 1000000
```

## 期待される結果

`n` を大きくするほど合計はゆっくり増える (調和級数). 例:

```
sum of 1/k (k=1..100) = 5.187378
```

確保を書く前に実行すると, 異常終了する (メモリを確保していないため). 引数 `n` を変えて値が変わることも確かめよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ array_dynamic.cpp
#include <cstdio>
#include <cstdlib>

int main(int argc, char ** argv) {
  /* 要素数 n を実行時 (コマンドライン引数) で決める */
  long n = (argc > 1 ? atol(argv[1]) : 100);
  double * a = NULL;
  // TODO: a に double n 個分の領域を malloc で確保せよ.
  for (long i = 0; i < n; i++) {
    a[i] = 1.0 / (i + 1);   /* 1/1, 1/2, 1/3, ... */
  }
  double s = 0.0;
  for (long i = 0; i < n; i++) {
    s += a[i];
  }
  printf("sum of 1/k (k=1..%ld) = %f\n", n, s);
  free(a);                  /* 確保した領域は解放する */
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore array_dynamic.cpp -o array_dynamic_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./array_dynamic_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ array_dynamic.f90
program array_dynamic
  character(len=32) :: arg
  integer(8) :: n, i
  real(8), allocatable :: a(:)
  real(8) :: s
  ! 要素数 n を実行時 (コマンドライン引数) で決める
  n = 100
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  ! TODO: a に n 要素分の領域を確保せよ (allocate).
  do i = 1, n
     a(i) = 1.0d0 / i        ! 1/1, 1/2, 1/3, ...
  end do
  s = 0.0d0
  do i = 1, n
     s = s + a(i)
  end do
  print "(a,i0,a,f0.6)", "sum of 1/k (k=1..", n, ") = ", s
  deallocate(a)             ! 確保した領域は解放する
end program array_dynamic

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore array_dynamic.f90 -o array_dynamic_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./array_dynamic_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:array_dynamic.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:array_dynamic.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:array_dynamic.cpp}